In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install impyute
!pip install catboost
!pip install optuna
!pip install BayesianOptimization
!pip install bayesian-optimization

In [ ]:
# Data Wrangling
import pandas as pd
from pandas import Series, DataFrame
import numpy as np

# Visualization
import matplotlib.pylab as plt
from matplotlib import font_manager, rc
import seaborn as sns
%matplotlib inline

# Preprocessing & Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile
from impyute.imputation.cs import fast_knn
from sklearn.utils import all_estimators
from sklearn.metrics import r2_score,mean_squared_error
# from pycaret.classification import *

# Encoder

# Hyperparameter Optimization
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Modeling
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.base import ClassifierMixin
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import optuna
from sklearn import ensemble
from sklearn import metrics
from sklearn import model_selection
from bayes_opt import BayesianOptimization

# Evaluation
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

# Utility
import os
import time
import random
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image
import pickle
from tqdm import tqdm, tqdm_notebook
import platform
from itertools import combinations
from scipy.stats.mstats import gmean

#matplotlib 한글깨짐 지원
import platform

path = "c:/Windows/Fonts/malgun.ttf"
if platform.system() == 'Darwin':
    rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    font_name = font_manager.FontProperties(fname=path).get_name()
    rc('font', family=font_name)
else:
    print('Unknown system...')
rc('axes', unicode_minus=False) 

### Read Data

In [ ]:
train = pd.read_csv('data/머러컴피티션/2등cat/best_snd_scale_shap/snd_sum_shap_0615.csv')
test = pd.read_csv('data/머러컴피티션/2등cat/best_snd_scale_shap/snd_sum_shap_te_0615.csv')

In [ ]:
ID_test = pd.DataFrame({'custid': pd.read_csv('data/머러컴피티션/X_test.csv', encoding='cp949').custid.unique()})
target = pd.read_csv('data/머러컴피티션/y_train.csv', encoding = "cp949").group

In [ ]:
train.to_csv('snd_shap.0615.csv')
test.to_csv('snd_shap_te.0615.csv')

In [ ]:
target

In [ ]:
ID_test

## Build Models

In [ ]:
train = train.iloc[:,1:]
test = test.iloc[:,1:]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.3, random_state = 0)

In [ ]:
X_val

In [ ]:
import optuna

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
import sklearn

In [ ]:
bayesian_params = {
    'max_depth':(8, 16),
    'num_leaves':(24, 64),
    'min_child_samples':(10, 200),
    'min_child_weight':(1, 50),
    'subsample':(0.5, 1),
    'colsample_bytree':(0.5, 1),
    'max_bin':(10, 500),
    'reg_lambda':(0.001, 10),
    'reg_alpha':(0.01, 50)
}

In [ ]:
def lgb_rmse_eval(max_depth, num_leaves, min_child_samples, min_child_weight, subsample, 
                colsample_bytree, max_bin, reg_lambda, reg_alpha):
    
    params = {
        "n_estimators":2000, 
        "learning_rate":0.02,
        'max_depth':int(round(max_depth)),
        'num_leaves':int(round(num_leaves)),
        'min_child_samples': int(round(min_child_samples)),
        'min_child_weight': int(round(min_child_weight)),
        'subsample':max(min(subsample, 1), 0),
        'colsample_bytree':max(min(colsample_bytree, 1), 0),
        'reg_lambda': max(reg_lambda,0),
        'reg_alpha': max(reg_alpha, 0),
        'application': 'multiclass',
        'objective': 'multiclass',
        'metric': 'multi_logloss',
        'num_classes' : 8,
        'n_jobs' : -1
    }
    
    lgb_model = LGBMClassifier(**params)
    lgb_model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)],eval_metric= 'multi_logloss', verbose= 100, 
                early_stopping_rounds= 100)
    valid_pred = lgb_model.predict_proba(X_val)
    logloss = sklearn.metrics.log_loss(y_val, valid_pred)
    
    return logloss

In [ ]:
lgbBO = BayesianOptimization(f=lgb_rmse_eval, pbounds=bayesian_params, random_state=1000)
lgbBO.maximize(init_points=5, n_iter=10)

In [ ]:
# dictionary에 있는 target값을 모두 추출
target_list = []
for result in lgbBO.res:
    target = result['target']
    target_list.append(target)
print(target_list)
# 가장 큰 target 값을 가지는 순번(index)를 추출
print('maximum target index:', np.argmin(np.array(target_list)))

In [ ]:
# 가장 큰 target값을 가지는 index값을 기준으로 res에서 해당 parameter 추출. 
max_dict = lgbBO.res[np.argmin(np.array(target_list))]
print(max_dict)

### Kfold

In [ ]:
X_new = train
X_te_new = test

In [ ]:
targets = pd.concat([target,target])

In [ ]:
'target': 1.4684193341335265, 'params': {'colsample_bytree': 0.5141043094390911, 'max_bin': 254.3407418874025, 'max_depth': 15.332770159742921, 'min_child_samples': 103.21393354412879, 'min_child_weight': 46.90043392010393, 'num_leaves': 32.33623155988221, 'reg_alpha': 12.21846097676822, 'reg_lambda': 9.777330191074164, 'subsample': 0.6988643735876281}}

In [ ]:
from sklearn.model_selection import KFold
ftr = X_new
target = target
test_proba = []

def train_apps_all_with_oof(ftr, ttmp_arget, nfolds=5):
    ftr = ftr
    tmp_target = target

    # nfolds 개의 cross validatin fold set을 가지는 KFold 생성 
    folds = KFold(n_splits = nfolds, shuffle=True, random_state=0)
    
    # Out of Folds로 학습된 모델의 validation set을 예측하여 결과 확률을 담을 array 생성.
    # validation set가 n_split갯수만큼 있으므로 크기는 ftr_app의 크기가 되어야 함. 
    oof_preds = np.zeros((ftr.shape[0],))  
    
    # Ouf of Folds로 학습된 모델의 test dataset을 예측하여 결과 확률을 담을 array 생성. 
    test_preds = np.zeros(((X_te_new.shape[0],)))
    
    # n_estimators를 4000까지 확대. 
    clf = LGBMClassifier(
                nthread=4,
                n_estimators=4000,
                learning_rate=0.01,
                max_depth=16,
                num_leaves=32,
                colsample_bytree=0.5141,
                subsample=0.698,
                max_bin=255,
                reg_alpha=12.218,
                reg_lambda=9.777,
                min_child_weight=47,
                min_child_samples=103,
                silent=-1,
                verbose=-1,
                )

    # nfolds 번 cross validation Iteration 반복하면서 OOF 방식으로 학습 및 테스트 데이터 예측
    for fold_idx, (train_idx, valid_idx) in enumerate(folds.split(ftr)):
        print('##### iteration ', fold_idx, ' 시작')
        # 학습용 데이터 세트의 인덱스와 검증용 데이터 세트의 인덱스 추출하여 이를 기반으로 학습/검증 데이터 추출
        train_x = ftr.iloc[train_idx, :]
        train_y = target.iloc[train_idx]
        valid_x = ftr.iloc[valid_idx, :]
        valid_y = target.iloc[valid_idx]
        
        # 추출된 학습/검증 데이터 세트로 모델 학습. early_stopping은 200으로 증가. 
        clf.fit(train_x, train_y, eval_set=[(train_x, train_y), (valid_x, valid_y)], eval_metric= 'multi_logloss', verbose= 200, 
                early_stopping_rounds= 200)
        # 검증 데이터 세트로 예측된 확률 저장. 사용되지는 않음. 
        #oof_preds[valid_idx] = clf.predict(valid_x, num_iteration=clf.best_iteration_)       
        # 학습된 모델로 테스트 데이터 세트에 예측 확률 계산. 
        # nfolds 번 반복 실행하므로 평균 확률을 구하기 위해 개별 수행시 마다 수행 횟수로 나눈 확률을 추후에 더해서 최종 평균 확률 계산. 
        test_proba.append( clf.predict_proba(test, num_iteration=clf.best_iteration_)/folds.n_splits)
        
        
    return clf, test_proba

In [ ]:
# 튜닝 후
clf, test_proba = train_apps_all_with_oof(ftr, target, nfolds=5)

In [ ]:
# 기존튜닝
clf, test_proba = train_apps_all_with_oof(ftr, target, nfolds=5)

In [ ]:
test_mean_proba = (test_proba[0] + test_proba[1] + test_proba[2] + test_proba[3] + test_proba[4])

In [ ]:
pd.DataFrame(test_mean_proba)

In [ ]:
len(test_proba[0])

## Deployment

In [ ]:
ID_test

In [ ]:
t = pd.Timestamp.now()
fname = f"lgbm_submission_{t.month:02}{t.day:02}{t.hour:02}{t.minute:02}.csv"
pred = pd.DataFrame(test_mean_proba)
pred.columns = ['F20','F30','F40','F50','M20','M30','M40','M50']
submissions = pd.concat([pd.Series(ID_test.custid, name = 'ID'), pred], axis = 1)
submissions.to_csv(fname, index=False)
print(f"'{fname}' is ready to submit.")